In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp.silver
COMMENT 'Camada Silver: dados limpos, deduplicados e conformados para modelagem analítica';

## mvp.silver.pix_transacoes

> Limpeza de nulos em atributos de perfil, unifica nomes das regiões brasileiras e cria colunas temporais de suporte.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS mvp.silver.pix_transacoes (
    anomes                                  INT,
    data_competencia                        DATE,
    ano                                     INT,
    mes                                     INT,
    pag_pfpj                                STRING,
    rec_pfpj                                STRING,
    pag_regiao                              STRING,
    rec_regiao                              STRING,
    pag_faixa_etaria                        STRING,
    rec_faixa_etaria                        STRING,
    forma_iniciacao                         STRING,
    natureza                                STRING,
    finalidade                              STRING,
    montante_reais                          DECIMAL(18, 2),
    quantidade_tx                           BIGINT,
    _silver_processed_at                    TIMESTAMP
)
PARTITIONED BY (anomes)
COMMENT 'Tabela Silver de Transações Pix deduplicada e padronizada';

MERGE INTO mvp.silver.pix_transacoes AS target
USING (
    SELECT 
        anomes,
        TO_DATE(CONCAT(CAST(anomes AS STRING), '01'), 'yyyyMMdd') AS data_competencia,
        CAST(SUBSTRING(CAST(anomes AS STRING), 1, 4) AS INT)      AS ano,
        CAST(SUBSTRING(CAST(anomes AS STRING), 5, 2) AS INT)      AS mes,
        COALESCE(NULLIF(pag_pfpj, ''), 'NÃO INFORMADO')            AS pag_pfpj,
        COALESCE(NULLIF(rec_pfpj, ''), 'NÃO INFORMADO')            AS rec_pfpj,
        CASE 
            WHEN pag_regiao LIKE '%CENTRO%' THEN 'CENTRO-OESTE'
            WHEN pag_regiao IS NULL OR pag_regiao = '' THEN 'NÃO INFORMADO'
            ELSE pag_regiao
        END AS pag_regiao,
        CASE 
            WHEN rec_regiao LIKE '%CENTRO%' THEN 'CENTRO-OESTE'
            WHEN rec_regiao IS NULL OR rec_regiao = '' THEN 'NÃO INFORMADO'
            ELSE rec_regiao
        END AS rec_regiao,
        COALESCE(NULLIF(pag_idade, ''), 'NÃO INFORMADO')           AS pag_faixa_etaria,
        COALESCE(NULLIF(rec_idade, ''), 'NÃO INFORMADO')           AS rec_faixa_etaria,
        COALESCE(NULLIF(forma_iniciacao, ''), 'OUTRA')             AS forma_iniciacao,
        COALESCE(NULLIF(natureza, ''), 'OUTRA')                    AS natureza,
        COALESCE(NULLIF(finalidade, ''), 'OUTRA')                  AS finalidade,
        montante_reais,
        quantidade_tx,
        current_timestamp()                                        AS _silver_processed_at
    FROM mvp.bronze.pix_transacoes
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY 
            anomes,
            pag_pfpj,
            rec_pfpj,
            pag_regiao,
            rec_regiao,
            pag_idade,
            rec_idade,
            forma_iniciacao,
            natureza,
            finalidade
        ORDER BY _ingestion_timestamp DESC
    ) = 1
) AS source
ON target.anomes = source.anomes
   AND target.pag_pfpj <=> source.pag_pfpj
   AND target.rec_pfpj <=> source.rec_pfpj
   AND target.pag_regiao <=> source.pag_regiao
   AND target.rec_regiao <=> source.rec_regiao
   AND target.pag_faixa_etaria <=> source.pag_faixa_etaria
   AND target.rec_faixa_etaria <=> source.rec_faixa_etaria
   AND target.forma_iniciacao <=> source.forma_iniciacao
   AND target.natureza <=> source.natureza
   AND target.finalidade <=> source.finalidade
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
45997,0,0,45997


> Inclusão do dicionário regulatório do Bacen para as formas de iniciação do Pix.

In [0]:
%sql
-- 1. Adiciona as colunas no schema da tabela Silver existente
ALTER TABLE mvp.silver.pix_transacoes 
ADD COLUMNS (
    forma_iniciacao_desc STRING,
    ordem_iniciacao      INT
);

-- 2. Atualiza os dados da Silver aplicando o dicionário regulatório
UPDATE mvp.silver.pix_transacoes
SET 
    forma_iniciacao_desc = CASE 
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('DICT', 'CHAVE') 
            THEN 'Chave Pix (Endereçamento via CPF/CNPJ, E-mail, Celular ou EVP)'
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('MANU', 'INSERCAO_MANUAL', 'MANUAL') 
            THEN 'Inserção Manual (Dados de Agência e Conta)'
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('QRDN', 'QR_CODE_DINAMICO', 'DYNAMIC_QR', 'DINAMICO') 
            THEN 'QR Code Dinâmico (Cobrança com Valor e Conciliação no Varejo/API)'
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('QRES', 'QR_CODE_ESTATICO', 'STATIC_QR', 'ESTATICO') 
            THEN 'QR Code Estático (Ponto de Venda Físico / P2P com Chave Fixa)'
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('INIC', 'INICIADOR_PAGAMENTO', 'OPEN_FINANCE', 'ITP') 
            THEN 'Iniciador de Pagamento (Open Finance / Sem Redirecionamento Manual)'
        ELSE 'Outras Modalidades de Iniciação'
    END,
    ordem_iniciacao = CASE 
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('DICT', 'CHAVE') THEN 1
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('MANU', 'INSERCAO_MANUAL', 'MANUAL') THEN 2
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('QRDN', 'QR_CODE_DINAMICO', 'DYNAMIC_QR', 'DINAMICO') THEN 3
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('QRES', 'QR_CODE_ESTATICO', 'STATIC_QR', 'ESTATICO') THEN 4
        WHEN UPPER(TRIM(forma_iniciacao)) IN ('INIC', 'INICIADOR_PAGAMENTO', 'OPEN_FINANCE', 'ITP') THEN 5
        ELSE 6
    END;

num_affected_rows
45997


In [0]:
%sql
-- ==============================================================================
-- AUDITORIA DE QUALIDADE E CONFORMIDADE: mvp.silver.pix_transacoes
-- ==============================================================================
WITH auditoria_temporal_e_nulos AS (
    SELECT 
        anomes,
        COUNT(1)                                                AS total_linhas,
        
        -- 1. Consistência de Datas
        SUM(CASE 
            WHEN data_competencia != TO_DATE(CONCAT(CAST(anomes AS STRING), '01'), 'yyyyMMdd') 
              OR ano != CAST(SUBSTRING(CAST(anomes AS STRING), 1, 4) AS INT)
              OR mes != CAST(SUBSTRING(CAST(anomes AS STRING), 5, 2) AS INT)
            THEN 1 ELSE 0 
        END)                                                    AS qtd_inconsistencias_data,
        
        -- 2. Verificação de Nulos Reais e Fallbacks
        SUM(CASE WHEN pag_pfpj IS NULL OR pag_pfpj IN ('', 'NÃO INFORMADO') THEN 1 ELSE 0 END)         AS sem_info_pag_pfpj,
        SUM(CASE WHEN rec_pfpj IS NULL OR rec_pfpj IN ('', 'NÃO INFORMADO') THEN 1 ELSE 0 END)         AS sem_info_rec_pfpj,
        SUM(CASE WHEN pag_regiao IS NULL OR pag_regiao IN ('', 'NÃO INFORMADO') THEN 1 ELSE 0 END)     AS sem_info_pag_regiao,
        SUM(CASE WHEN rec_regiao IS NULL OR rec_regiao IN ('', 'NÃO INFORMADO') THEN 1 ELSE 0 END)     AS sem_info_rec_regiao,
        SUM(CASE WHEN pag_faixa_etaria IS NULL OR pag_faixa_etaria IN ('', 'NÃO INFORMADO') THEN 1 ELSE 0 END) AS sem_info_pag_idade,
        SUM(CASE WHEN rec_faixa_etaria IS NULL OR rec_faixa_etaria IN ('', 'NÃO INFORMADO') THEN 1 ELSE 0 END) AS sem_info_rec_idade,
        SUM(CASE WHEN forma_iniciacao IS NULL OR forma_iniciacao IN ('', 'OUTRA') THEN 1 ELSE 0 END)   AS sem_info_forma_iniciacao,
        
        -- 3. Métricas Financeiras e Volumetria
        SUM(CASE WHEN montante_reais IS NULL OR montante_reais < 0 THEN 1 ELSE 0 END)                   AS montante_invalido,
        SUM(CASE WHEN quantidade_tx IS NULL OR quantidade_tx <= 0 THEN 1 ELSE 0 END)                    AS quantidade_tx_invalida,
        SUM(quantidade_tx)                                      AS total_transacoes,
        SUM(montante_reais)                                     AS montante_total_reais,
        MAX(_silver_processed_at)                               AS ultimo_processamento_silver
    FROM mvp.silver.pix_transacoes
    GROUP BY anomes
),
auditoria_duplicatas AS (
    SELECT 
        anomes,
        COUNT(1) AS linhas_duplicadas
    FROM (
        SELECT 
            anomes, pag_pfpj, rec_pfpj, pag_regiao, rec_regiao,
            pag_faixa_etaria, rec_faixa_etaria, forma_iniciacao,
            natureza, finalidade
        FROM mvp.silver.pix_transacoes
        GROUP BY 
            anomes, pag_pfpj, rec_pfpj, pag_regiao, rec_regiao,
            pag_faixa_etaria, rec_faixa_etaria, forma_iniciacao,
            natureza, finalidade
        HAVING COUNT(1) > 1
    )
    GROUP BY anomes
)
SELECT 
    t.anomes,
    t.total_linhas,
    
    -- Status Geral de Qualidade da Partição
    CASE 
        WHEN COALESCE(d.linhas_duplicadas, 0) > 0 THEN '❌ Duplicidade Detectada'
        WHEN t.qtd_inconsistencias_data > 0        THEN '❌ Erro de Consistência Temporal'
        WHEN t.montante_invalido > 0 OR t.quantidade_tx_invalida > 0 THEN '❌ Valores Numéricos Inválidos'
        WHEN (t.sem_info_pag_pfpj + t.sem_info_rec_pfpj + t.sem_info_pag_regiao + t.sem_info_rec_regiao) > 0 
             THEN '⚠️ Alerta: Presença de "NÃO INFORMADO"'
        ELSE '✅ Conforme'
    END AS status_qualidade,
    
    COALESCE(d.linhas_duplicadas, 0) AS chaves_com_duplicacao,
    t.qtd_inconsistencias_data,
    (t.sem_info_pag_pfpj + t.sem_info_rec_pfpj + t.sem_info_pag_regiao + t.sem_info_rec_regiao + 
     t.sem_info_pag_idade + t.sem_info_rec_idade + t.sem_info_forma_iniciacao) AS total_registros_nao_informados,
    t.total_transacoes,
    t.montante_total_reais,
    t.ultimo_processamento_silver

FROM auditoria_temporal_e_nulos t
LEFT JOIN auditoria_duplicatas d
    ON t.anomes = d.anomes
ORDER BY t.anomes;

anomes,total_linhas,status_qualidade,chaves_com_duplicacao,qtd_inconsistencias_data,total_registros_nao_informados,total_transacoes,montante_total_reais,ultimo_processamento_silver
202011,19,✅ Conforme,0,0,0,71010,53890784.93,2026-09-27T20:34:07.049Z
202012,29,✅ Conforme,0,0,0,524441,3293600107.40,2026-09-27T20:34:07.049Z
202101,30,✅ Conforme,0,0,0,1802153,817058283.67,2026-09-27T20:34:07.049Z
202102,35,✅ Conforme,0,0,10,532659,361005193.43,2026-09-27T20:34:07.049Z
202103,39,✅ Conforme,0,0,0,6377119,4232818311.21,2026-09-27T20:34:07.049Z
202104,22,✅ Conforme,0,0,0,1652651,443519465.59,2026-09-27T20:34:07.049Z
202105,38,✅ Conforme,0,0,0,12566816,3478503140.57,2026-09-27T20:34:07.049Z
202106,134,✅ Conforme,0,0,0,5304238,1879406936.93,2026-09-27T20:34:07.049Z
202107,129,✅ Conforme,0,0,16,13970167,4966590578.59,2026-09-27T20:34:07.049Z
202108,130,✅ Conforme,0,0,0,8740037,3218501306.42,2026-09-27T20:34:07.049Z


## mvp.silver.pix_fraudes_med

> Gera data analítica, valida coerência dos valores e limpa métricas de devolução.

In [0]:
%sql
-- 1. Drop versão antiga com schema desalinhado
DROP TABLE IF EXISTS mvp.silver.pix_fraudes_med;

-- 2. Recriação da tabela com o schema definitivo
CREATE TABLE mvp.silver.pix_fraudes_med (
    anomes                                  INT,
    data_competencia                        DATE,
    ano                                     INT,
    mes                                     INT,
    qtde_pix_contestados                    BIGINT,
    qtde_contestacoes_aceitas               BIGINT,
    qtde_contestacoes_rejeitadas            BIGINT,
    taxa_aceite_por_100k                    DECIMAL(12, 4),
    qtde_usuarios_marcados_fraude           BIGINT,
    qtde_chaves_marcadas_fraude             BIGINT,
    valor_contestacoes_aceitas_reais        DECIMAL(18, 2),
    qtde_devolucoes_integrais               BIGINT,
    valor_devolvido_integral_reais          DECIMAL(18, 2),
    qtde_devolucoes_parciais                BIGINT,
    valor_devolvido_parcial_reais           DECIMAL(18, 2),
    valor_residual_nao_devolvido_reais      DECIMAL(18, 2),
    qtde_nao_devolvida_saldo_insuficiente   BIGINT,
    valor_nao_devolvido_saldo_insuficiente  DECIMAL(18, 2),
    qtde_nao_devolvida_conta_encerrada      BIGINT,
    valor_nao_devolvido_conta_encerrada     DECIMAL(18, 2),
    qtde_nao_devolvida_motivos_diversos     BIGINT,
    valor_nao_devolvido_motivos_diversos    DECIMAL(18, 2),
    percentual_devolucao                    DECIMAL(6, 2),
    qtde_bloqueio_cautelar_liberados        BIGINT,
    valor_bloqueio_cautelar_liberado_reais  DECIMAL(18, 2),
    qtde_bloqueio_cautelar_devolvidos       BIGINT,
    valor_bloqueio_cautelar_devolvido_reais DECIMAL(18, 2),
    taxa_efetiva_recuperacao_pct            DECIMAL(10, 2),
    _silver_processed_at                    TIMESTAMP
)
PARTITIONED BY (anomes)
COMMENT 'Tabela Silver de Fraudes MED conformada e enriquecida temporalmente';

-- 3. Execução MERGE INTO
MERGE INTO mvp.silver.pix_fraudes_med AS target
USING (
    SELECT 
        anomes,
        TO_DATE(CONCAT(CAST(anomes AS STRING), '01'), 'yyyyMMdd') AS data_competencia,
        CAST(SUBSTRING(CAST(anomes AS STRING), 1, 4) AS INT)      AS ano,
        CAST(SUBSTRING(CAST(anomes AS STRING), 5, 2) AS INT)      AS mes,
        qtde_pix_contestados,
        qtde_contestacoes_aceitas,
        qtde_contestacoes_rejeitadas,
        taxa_aceite_por_100k,
        qtde_usuarios_marcados_fraude,
        qtde_chaves_marcadas_fraude,
        valor_contestacoes_aceitas_reais,
        qtde_devolucoes_integrais,
        valor_devolvido_integral_reais,
        qtde_devolucoes_parciais,
        valor_devolvido_parcial_reais,
        valor_residual_nao_devolvido_reais,
        qtde_nao_devolvida_saldo_insuficiente,
        valor_nao_devolvido_saldo_insuficiente,
        qtde_nao_devolvida_conta_encerrada,
        valor_nao_devolvido_conta_encerrada,
        qtde_nao_devolvida_motivos_diversos,
        valor_nao_devolvido_motivos_diversos,
        percentual_devolucao,
        qtde_bloqueio_cautelar_liberados,
        valor_bloqueio_cautelar_liberado_reais,
        qtde_bloqueio_cautelar_devolvidos,
        valor_bloqueio_cautelar_devolvido_reais,
        ROUND(
            (valor_devolvido_integral_reais / NULLIF(valor_contestacoes_aceitas_reais, 0)) * 100, 
            2
        ) AS taxa_efetiva_recuperacao_pct,
        current_timestamp() AS _silver_processed_at
    FROM mvp.bronze.pix_fraudes_med
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY anomes 
        ORDER BY _ingestion_timestamp DESC
    ) = 1
) AS source
ON target.anomes = source.anomes
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
52,0,0,52


In [0]:
%sql
-- ==============================================================================
-- AUDITORIA DE QUALIDADE E CONFORMIDADE: mvp.silver.pix_fraudes_med
-- ==============================================================================
WITH auditoria_metricas AS (
    SELECT 
        s.anomes,
        s.data_competencia,
        s.ano,
        s.mes,
        
        -- 1. Verificação de Integridade Temporal
        CASE 
            WHEN s.data_competencia != TO_DATE(CONCAT(CAST(s.anomes AS STRING), '01'), 'yyyyMMdd')
              OR s.ano != CAST(SUBSTRING(CAST(s.anomes AS STRING), 1, 4) AS INT)
              OR s.mes != CAST(SUBSTRING(CAST(s.anomes AS STRING), 5, 2) AS INT)
            THEN 1 ELSE 0 
        END AS flag_erro_temporal,

        -- 2. Verificação de Duplicidade
        COUNT(1) OVER (PARTITION BY s.anomes) AS linhas_por_anomes,

        -- 3. Validação Matemática da Taxa Efetiva de Recuperação
        CASE 
            WHEN s.valor_contestacoes_aceitas_reais IS NULL OR s.valor_contestacoes_aceitas_reais = 0 THEN 0
            WHEN ABS(
                s.taxa_efetiva_recuperacao_pct - 
                ROUND((s.valor_devolvido_integral_reais / s.valor_contestacoes_aceitas_reais) * 100, 2)
            ) > 0.01 THEN 1
            ELSE 0
        END AS flag_erro_taxa_recuperacao,

        -- 4. Validação Contábil Realista: O montante devolvido não pode superar o total aceito
        CASE 
            WHEN (COALESCE(s.valor_devolvido_integral_reais, 0) + COALESCE(s.valor_devolvido_parcial_reais, 0)) > 
                 (s.valor_contestacoes_aceitas_reais + 1.00) THEN 1
            ELSE 0 
        END AS flag_inconsistencia_contabil_grave,

        -- 5. Validação de Nulos em Campos Mandatórios
        CASE 
            WHEN s.anomes IS NULL 
              OR s.qtde_pix_contestados IS NULL 
              OR s.qtde_contestacoes_aceitas IS NULL 
              OR s.valor_contestacoes_aceitas_reais IS NULL
            THEN 1 ELSE 0 
        END AS flag_nulos_criticos,

        s.qtde_pix_contestados,
        s.qtde_contestacoes_aceitas,
        s.valor_contestacoes_aceitas_reais,
        s.valor_devolvido_integral_reais,
        s.taxa_efetiva_recuperacao_pct,
        s._silver_processed_at

    FROM mvp.silver.pix_fraudes_med s
),
reconciliacao_bronze AS (
    SELECT 
        anomes,
        valor_contestacoes_aceitas_reais AS bronze_valor_aceito,
        qtde_pix_contestados            AS bronze_qtde_contestada
    FROM mvp.bronze.pix_fraudes_med
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY anomes 
        ORDER BY _ingestion_timestamp DESC
    ) = 1
)
SELECT 
    m.anomes,
    m.data_competencia,
    
    -- Status Consolidado de Qualidade
    CASE 
        WHEN m.linhas_por_anomes > 1               THEN '❌ Duplicidade de Competência'
        WHEN m.flag_erro_temporal = 1             THEN '❌ Inconsistência Temporal'
        WHEN m.flag_nulos_criticos = 1            THEN '❌ Nulos em Campos Chave'
        WHEN m.flag_erro_taxa_recuperacao = 1     THEN '❌ Erro no Cálculo da Taxa (%)'
        WHEN m.valor_contestacoes_aceitas_reais != b.bronze_valor_aceito THEN '❌ Divergência vs Bronze'
        WHEN m.flag_inconsistencia_contabil_grave = 1 THEN '❌ Valor Devolvido Excede o Aceito'
        ELSE '✅ Conforme'
    END AS status_auditoria,
    
    m.qtde_pix_contestados,
    m.qtde_contestacoes_aceitas,
    m.valor_contestacoes_aceitas_reais,
    m.valor_devolvido_integral_reais,
    m.taxa_efetiva_recuperacao_pct,
    (m.valor_contestacoes_aceitas_reais - b.bronze_valor_aceito) AS dif_reais_bronze_silver,
    m._silver_processed_at

FROM auditoria_metricas m
LEFT JOIN reconciliacao_bronze b
    ON m.anomes = b.anomes
ORDER BY m.anomes;

anomes,data_competencia,status_auditoria,qtde_pix_contestados,qtde_contestacoes_aceitas,valor_contestacoes_aceitas_reais,valor_devolvido_integral_reais,taxa_efetiva_recuperacao_pct,dif_reais_bronze_silver,_silver_processed_at
202201,2022-01-01,✅ Conforme,71508,57777,122558868.28,4261762.10,3.48,0.00,2026-09-27T19:22:10.929Z
202202,2022-02-01,✅ Conforme,78636,60680,136034158.58,5355273.06,3.94,0.00,2026-09-27T19:22:10.929Z
202203,2022-03-01,✅ Conforme,134964,105016,216939933.12,10998092.06,5.07,0.00,2026-09-27T19:22:10.929Z
202204,2022-04-01,✅ Conforme,152636,114763,178163735.34,8494563.65,4.77,0.00,2026-09-27T19:22:10.929Z
202205,2022-05-01,✅ Conforme,229401,149006,224862932.69,8179939.00,3.64,0.00,2026-09-27T19:22:10.929Z
202206,2022-06-01,✅ Conforme,190206,146526,223677609.87,15034322.97,6.72,0.00,2026-09-27T19:22:10.929Z
202207,2022-07-01,✅ Conforme,186056,142493,213715054.18,9793327.15,4.58,0.00,2026-09-27T19:22:10.929Z
202208,2022-08-01,✅ Conforme,218131,155021,233907823.75,10759053.55,4.60,0.00,2026-09-27T19:22:10.929Z
202209,2022-09-01,✅ Conforme,196377,137732,220824321.90,12832651.52,5.81,0.00,2026-09-27T19:22:10.929Z
202210,2022-10-01,✅ Conforme,199364,147113,232190741.27,15522803.17,6.69,0.00,2026-09-27T19:22:10.929Z


## mvp.silver.ibge_populacao

> Padroniza o cadastro geográfico para cruzamento referencial com as regiões do Bacen.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS mvp.silver.ibge_populacao (
    id_regiao_ibge          INT,
    nome_regiao             STRING,
    ano_referencia          INT,
    populacao_residente     BIGINT,
    _silver_processed_at    TIMESTAMP
)
COMMENT 'Tabela Silver de População por Região conformada com o IBGE SIDRA';

MERGE INTO mvp.silver.ibge_populacao AS target
USING (
    SELECT 
        id_regiao_ibge,
        CASE 
            WHEN nome_regiao LIKE '%CENTRO%' THEN 'CENTRO-OESTE'
            ELSE nome_regiao
        END AS nome_regiao,
        ano_referencia,
        populacao_residente,
        current_timestamp() AS _silver_processed_at
    FROM mvp.bronze.ibge_populacao
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY id_regiao_ibge, ano_referencia 
        ORDER BY _ingestion_timestamp DESC
    ) = 1
) AS source
ON target.id_regiao_ibge = source.id_regiao_ibge
   AND target.ano_referencia = source.ano_referencia
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
-- ==============================================================================
-- AUDITORIA DE QUALIDADE E CONFORMIDADE: mvp.silver.ibge_populacao
-- ==============================================================================
WITH auditoria_silver AS (
    SELECT 
        s.ano_referencia,
        COUNT(1)                                                AS total_regioes_cadastradas,
        
        -- 1. Unicidade de Grão (id_regiao_ibge + ano_referencia)
        COUNT(1) - COUNT(DISTINCT s.id_regiao_ibge)             AS qtd_chaves_duplicadas,
        
        -- 2. Conformidade Cadastral das 5 Grandes Regiões do Brasil
        SUM(CASE 
            WHEN s.id_regiao_ibge = 1 AND s.nome_regiao = 'NORTE' THEN 0
            WHEN s.id_regiao_ibge = 2 AND s.nome_regiao = 'NORDESTE' THEN 0
            WHEN s.id_regiao_ibge = 3 AND s.nome_regiao = 'SUDESTE' THEN 0
            WHEN s.id_regiao_ibge = 4 AND s.nome_regiao = 'SUL' THEN 0
            WHEN s.id_regiao_ibge = 5 AND s.nome_regiao = 'CENTRO-OESTE' THEN 0
            ELSE 1 
        END)                                                    AS qtd_regioes_fora_padrao_ibge,
        
        -- 3. Validação de Nulos e Strings Vazias
        SUM(CASE WHEN s.id_regiao_ibge IS NULL THEN 1 ELSE 0 END)                     AS nulos_id_regiao,
        SUM(CASE WHEN s.nome_regiao IS NULL OR s.nome_regiao = '' THEN 1 ELSE 0 END) AS nulos_nome_regiao,
        SUM(CASE WHEN s.populacao_residente IS NULL THEN 1 ELSE 0 END)                AS nulos_populacao,
        
        -- 4. Consistência Demográfica (valores positivos e plausíveis)
        SUM(CASE 
            WHEN s.populacao_residente <= 0 
              OR s.populacao_residente < 5000000 
              OR s.populacao_residente > 120000000 
            THEN 1 ELSE 0 
        END)                                                    AS qtd_populacao_implausivel,
        
        SUM(s.populacao_residente)                              AS populacao_total_brasil,
        MAX(s._silver_processed_at)                             AS ultimo_processamento_silver
    FROM mvp.silver.ibge_populacao s
    GROUP BY s.ano_referencia
),
-- Deduplicação prévia da Bronze em subquery isolada para evitar conflito com GROUP BY
bronze_deduplicada AS (
    SELECT 
        id_regiao_ibge,
        ano_referencia,
        populacao_residente
    FROM mvp.bronze.ibge_populacao
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY id_regiao_ibge, ano_referencia 
        ORDER BY _ingestion_timestamp DESC
    ) = 1
),
reconciliacao_bronze AS (
    SELECT 
        ano_referencia,
        COUNT(DISTINCT id_regiao_ibge) AS bronze_total_regioes,
        SUM(populacao_residente)       AS bronze_populacao_total
    FROM bronze_deduplicada
    GROUP BY ano_referencia
)
SELECT 
    a.ano_referencia,
    a.total_regioes_cadastradas,
    
    -- Status Consolidado
    CASE 
        WHEN a.total_regioes_cadastradas != 5       THEN '❌ Faltam ou Sobram Regiões (Esperado: 5)'
        WHEN a.qtd_chaves_duplicadas > 0           THEN '❌ Duplicidade de Chave'
        WHEN a.qtd_regioes_fora_padrao_ibge > 0     THEN '❌ Nome/ID Fora do Padrão IBGE'
        WHEN (a.nulos_id_regiao + a.nulos_nome_regiao + a.nulos_populacao) > 0 THEN '❌ Presença de Nulos'
        WHEN a.qtd_populacao_implausivel > 0       THEN '⚠️ Valores Demográficos Anômalos'
        WHEN a.populacao_total_brasil != b.bronze_populacao_total THEN '❌ Divergência vs Bronze'
        ELSE '✅ Conforme'
    END AS status_auditoria,
    
    a.populacao_total_brasil,
    b.bronze_populacao_total,
    (a.populacao_total_brasil - b.bronze_populacao_total) AS dif_populacao_bronze_silver,
    a.ultimo_processamento_silver

FROM auditoria_silver a
LEFT JOIN reconciliacao_bronze b
    ON a.ano_referencia = b.ano_referencia
ORDER BY a.ano_referencia;

ano_referencia,total_regioes_cadastradas,status_auditoria,populacao_total_brasil,bronze_populacao_total,dif_populacao_bronze_silver,ultimo_processamento_silver
2022,5,✅ Conforme,203080756,203080756,0,2026-09-27T19:22:24.912Z
